# XMF-GNN — Graph Construction Pipeline

Mirrors the GNN4ID `GNN4ID.ipynb` workflow but with the XMF-GNN specific
feature extraction (multi-scale temporal block, edge-features-as-node-attrs,
single `connected_to` relation).

Paper references: Sec. 3.1 (Feature extraction), Sec. 3.2 (Graph-level
construction), Algorithm 1.

Pipeline:
1. **PCAP → CSV** (per-class) via `Feature_extractor_flow_packet_combined.py`.
2. **CSV enrichment** via `additional_features()` — adds the 4×13=52-dim
   multi-scale temporal block.
3. **Per-class filter + balance** (paper Sec. 4.2 step 1+2).
4. **CSV → HeteroData graphs** via `NIDSDataset`.

Set the paths below before running.

In [ ]:
import os, glob
from pathlib import Path

# Shared PCAP dataset (used across multiple projects)
RAW_PCAP_DIR    = os.path.expanduser('~/Tutay/Tutay_Sec/CIC_IoT_2023_PCAP')
# Project-specific outputs
PROJECT_ROOT    = os.path.dirname(os.path.abspath('__file__'))
DATA_ROOT       = os.path.join(PROJECT_ROOT, 'data')
OUT_CSV_DIR     = os.path.join(DATA_ROOT, 'Extracted_Flow_Features')
PROCESSED_ROOT  = os.path.join(DATA_ROOT, 'processed_xmfgnn')

Path(OUT_CSV_DIR).mkdir(parents=True, exist_ok=True)
Path(PROCESSED_ROOT).mkdir(parents=True, exist_ok=True)
print('PCAP dir:', RAW_PCAP_DIR)
print('CSV dir :', OUT_CSV_DIR)
print('Processed:', PROCESSED_ROOT)


## Step 1 — Run NFStream over each PCAP file

Same script as XG-NID. Runs once per PCAP; output is one CSV per PCAP.
Skip this section if you have already produced the CSVs.

In [ ]:
import os, glob
from pathlib import Path

# Shared PCAP dataset (used across multiple projects)
RAW_PCAP_DIR    = os.path.expanduser('~/Tutay/Tutay_Sec/CIC_IoT_2023_PCAP')
# Project-specific outputs
PROJECT_ROOT    = os.path.dirname(os.path.abspath('__file__'))
DATA_ROOT       = os.path.join(PROJECT_ROOT, 'data')
OUT_CSV_DIR     = os.path.join(DATA_ROOT, 'Extracted_Flow_Features')
PROCESSED_ROOT  = os.path.join(DATA_ROOT, 'processed_xmfgnn')

Path(OUT_CSV_DIR).mkdir(parents=True, exist_ok=True)
Path(PROCESSED_ROOT).mkdir(parents=True, exist_ok=True)
print('PCAP dir:', RAW_PCAP_DIR)
print('CSV dir :', OUT_CSV_DIR)
print('Processed:', PROCESSED_ROOT)


## Step 2 — Add multi-scale temporal features

Implements paper Algorithm 1: for each (destination IP × window size)
compute 13 statistical features (pkt_rate, byte_rate, avg_pkt_size,
syn/ack/fin_rst/icmp/udp ratios, src_dst_ratio, iat_mean/std,
unique_dst_ports, vuln_port_hits) over rolling windows of
{10s, 30s, 60s, 300s}. Concatenated → 52-dim block per flow.

In [ ]:
from Utility.Additional_Features import additional_features, DEFAULT_WINDOW_SIZES_SEC

for csv in glob.glob(os.path.join(OUT_CSV_DIR, '*.csv')):
    print('Enriching', csv)
    additional_features(csv, window_sizes_sec=DEFAULT_WINDOW_SIZES_SEC)

## Step 3 — Filter by attacker MAC, balance classes

Paper Sec. 4.2: bidirectional MAC filter, then 20,000 train / 4,000 test
per class via under/oversampling. See the
`Data_preprocessing_CIC-IoT2023.ipynb` notebook for the full preprocessing
pipeline -- this notebook just calls into the helpers.

In [ ]:
from Utility.Functions import split_csv, Combining_classes, DEFAULT_CIC_IOT2023_LABELS

for csv in glob.glob(os.path.join(OUT_CSV_DIR, '*.csv')):
    split_csv(csv, test_sample=4000, number_in_individual_class=20000)

Combining_classes(
    directory=OUT_CSV_DIR,
    classes_list=list(DEFAULT_CIC_IOT2023_LABELS.keys()),
    Number_in_individaul_class=20000,
    Number_of_test_samples=4000,
)

## Step 4 — Build HeteroData graphs

Paper Sec. 3.2: a single `connected_to` edge type, edge attributes folded
into node attributes. Run once for the train CSV, once for the test CSV.

In [ ]:
from Utility.Functions import NIDSDataset, DEFAULT_CIC_IOT2023_LABELS

TRAIN_CSV = os.path.join(OUT_CSV_DIR, 'train', 'df_class_8_train.csv')
TEST_CSV  = os.path.join(OUT_CSV_DIR, 'train', 'df_class_8_test.csv')

train_set = NIDSDataset(
    root=os.path.join(PROCESSED_ROOT, 'train'),
    label_dict=DEFAULT_CIC_IOT2023_LABELS,
    filename=[TRAIN_CSV],
    single_file=True,
)
test_set = NIDSDataset(
    root=os.path.join(PROCESSED_ROOT, 'test'),
    label_dict=DEFAULT_CIC_IOT2023_LABELS,
    filename=[TEST_CSV],
    single_file=True,
    test=True,
)
print('train graphs:', len(train_set), 'test graphs:', len(test_set))
print('Sample 0 metadata:', train_set[0].metadata())
print('Flow node x:', train_set[0]['flow'].x.shape)
print('Packet node x:', train_set[0]['packet'].x.shape)